# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Danil Shegai
**Student ID:S23069071**
**Group:202610:CSS4007-ENG-10**
**Repository:https://github.com/CardiganFlanagan/css4007-hw3**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> I used Claude (Anthropic) to help write the code in this notebook, to draft the Part A hand calculations, and to draft the written answers and a first proposal of the C2 ✓/✗ marks. I set up the keys, ran every cell myself, checked the Part A arithmetic against the check cell, checked every C2 mark against `reference`, and made sure every chunk id, score and token count quoted in the text is the one printed by my own run.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = **8.333**

**Query terms:** how, does, rag, retrieve, information (5 terms)

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.946 |
| does | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.946 |
| rag | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.946 |
| retrieve | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.946 |
| information | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.946 |

(The IDF has a value, but it is never used: every term is multiplied by f(t,d) = 0.)

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

> Every term contributes `idf(t) · 0 · 2.5 / (0 + 1.5·(…)) = 0`.
>
> - score(q, D1) = 0 + 0 + 0 + 0 + 0 = **0**
> - score(q, D2) = 0 + 0 + 0 + 0 + 0 = **0** (D2 has `retrieves` and `retrieval`, but not the exact token `retrieve`; `RAG` appears nowhere, because D2 spells it out as `Retrieval-Augmented Generation`)
> - score(q, D3) = 0 + 0 + 0 + 0 + 0 = **0**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 1 (tie) |
| D2 | 0 | 1 (tie) |
| D3 | 0 | 1 (tie) |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** 25 / 3 = **8.333** (stemming changes the tokens, not their number)

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.946 |
| doe | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.946 |
| rag | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.946 |
| retriev | 0 | **2** | 0 | **1** | ln((3 − 1 + 0.5) / (1 + 0.5)) = ln(2.5 / 1.5) = ln(1.667) = **0.5108** |
| inform | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = 1.946 |

**BM25 of each document**, worked:

> Only `retriev` in D2 contributes.
>
> Length normalisation for D2: 1 − b + b·|d|/avgdl = 0.25 + 0.75 · 10 / 8.333 = 0.25 + 0.90 = 1.15
>
> Denominator: f + k1 · 1.15 = 2 + 1.5 · 1.15 = 2 + 1.725 = 3.725
>
> Numerator: f · (k1 + 1) = 2 · 2.5 = 5
>
> - score(q, D2) = 0.5108 · 5 / 3.725 = 0.5108 · 1.3423 = **0.6857**
> - score(q, D1) = 0 (D1 has `gener`, but `gener` is not a query term)
> - score(q, D3) = 0

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tie) |
| D2 | 0.6857 | **1** |
| D3 | 0 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> **(a): none.** All three documents score exactly 0, so the "ranking" is just a three-way tie and the order is arbitrary. This is not an arithmetic mistake. BM25 counts exact token matches, and none of the five query tokens (`how`, `does`, `rag`, `retrieve`, `information`) occurs in any document. D2 is obviously the right document to a human, but it says `retrieves` / `retrieval` (not `retrieve`), and it spells out `Retrieval-Augmented Generation` where the query uses the abbreviation `RAG`.
>
> **(b): D2, with 0.6857.** Stemming maps `retrieve`, `retrieves` and `retrieval` all to `retriev`, so the query now matches D2 twice. The score is still small, because with only N = 3 documents a term that appears in one of them gets a low IDF (0.51). `rag` still matches nothing: stemming fixes word endings, not abbreviations or synonyms.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = **0.9644**

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = **0.78** | 0.9644 | √(0.49 + 0.01 + 0.16) = √0.66 = 0.8124 | 0.78 / (0.9644 · 0.8124) = 0.78 / 0.7835 = **0.9956** |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = **1.08** | 0.9644 | √(0.81 + 0.09 + 0.36) = √1.26 = 1.1225 | 1.08 / (0.9644 · 1.1225) = 1.08 / 1.0825 = **0.9977** |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = **0.34** | 0.9644 | √(0.01 + 0.64 + 0.04) = √0.69 = 0.8307 | 0.34 / (0.9644 · 0.8307) = 0.34 / 0.8011 = **0.4244** |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> **D2** (0.9977). D1 is very close behind (0.9956): the two vectors point in almost the same direction as the query, and D2 is only slightly closer.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> - **Against A1 (a):** no. BM25 gave a three-way tie at 0, so it had no ranking at all. Cosine gives a clear order D2 > D1 > D3.
> - **Against A1 (b):** the top document is the same (D2). But BM25 (b) cannot tell D1 and D3 apart (both 0), while cosine puts D1 far above D3 (0.9956 against 0.4244).

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 only sees whether the exact string occurs. The query says `RAG`, D2 says `Retrieval-Augmented Generation`: for BM25 those share no token at all. The query says `retrieve` and D2 says `retrieves` / `retrieval`, and without stemming these are three different tokens too. So BM25 (a) gives D2 zero. An embedding model maps text to a vector of meaning. It has learned that `RAG` and `Retrieval-Augmented Generation` mean the same thing, and that `retrieve information` is close to `retrieves relevant documents`. So D2 ends up close to the query even with no shared word. It also places D1 (language models answering questions) close, because the topic is related, even though D1 shares no query word either. That is why cosine ranks D1 second and BM25 does not.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
import math, re
import numpy as np

docs_A = {
    "D1": "Large language models can generate text and answer questions.",
    "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
    "D3": "Embeddings represent text as numerical vectors.",
}
query_A = "How does RAG retrieve information?"
STEMS = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
         "generate": "gener", "generating": "gener", "generation": "gener",
         "information": "inform", "does": "doe"}

def tok_A(s, stem=False):
    s = re.sub(r"[^\w\s-]", "", s.lower()).replace("-", " ")
    toks = s.split()
    return [STEMS.get(t, t) for t in toks] if stem else toks

def bm25_A(stem, k1=1.5, b=0.75):
    D = {k: tok_A(v, stem) for k, v in docs_A.items()}
    q = tok_A(query_A, stem)
    N, avgdl = len(D), sum(len(d) for d in D.values()) / len(D)
    print(f"--- stemming={stem}  query={q}  avgdl={avgdl:.3f}")
    for name, d in D.items():
        score = 0.0
        for t in q:
            n = sum(t in dd for dd in D.values())
            idf = math.log((N - n + 0.5) / (n + 0.5))
            f = d.count(t)
            part = idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(d) / avgdl))
            if f:
                print(f"  {name}: term={t!r} f={f} n={n} idf={idf:.4f} -> {part:.4f}")
            score += part
        print(f"  {name}: |d|={len(d)}  BM25={score:.4f}")

bm25_A(stem=False)
bm25_A(stem=True)

q = np.array([0.8, 0.2, 0.5])
for name, v in {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}.items():
    v = np.array(v)
    print(f"{name}: dot={q @ v:.4f} |q|={np.linalg.norm(q):.4f} |d|={np.linalg.norm(v):.4f} "
          f"cos={q @ v / (np.linalg.norm(q) * np.linalg.norm(v)):.4f}")

--- stemming=False  query=['how', 'does', 'rag', 'retrieve', 'information']  avgdl=8.333
  D1: |d|=9  BM25=0.0000
  D2: |d|=10  BM25=0.0000
  D3: |d|=6  BM25=0.0000
--- stemming=True  query=['how', 'doe', 'rag', 'retriev', 'inform']  avgdl=8.333
  D1: |d|=9  BM25=0.0000
  D2: term='retriev' f=2 n=1 idf=0.5108 -> 0.6857
  D2: |d|=10  BM25=0.6857
  D3: |d|=6  BM25=0.0000
D1: dot=0.7800 |q|=0.9644 |d|=0.8124 cos=0.9956
D2: dot=1.0800 |q|=0.9644 |d|=1.1225 cos=0.9977
D3: dot=0.3400 |q|=0.9644 |d|=0.8307 cos=0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import os, json, re
from pathlib import Path

# Keys: from .env locally, or from Colab Secrets. They are never printed.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
try:
    from google.colab import userdata  # only exists in Colab
    for k in ("OPENAI_API_KEY", "HF_TOKEN"):
        if not os.environ.get(k):
            try:
                os.environ[k] = userdata.get(k)
            except Exception:
                pass
except ImportError:
    pass
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))
print("HF_TOKEN set:      ", bool(os.environ.get("HF_TOKEN")))

DATA = Path("data")
corpus = [json.loads(line) for line in open(DATA / "corpus.jsonl", encoding="utf-8")]
by_id = {c["id"]: c for c in corpus}
queries = json.load(open(DATA / "queries.json", encoding="utf-8"))["queries"]
qa = json.load(open(DATA / "qa.json", encoding="utf-8"))["questions"]

print(len(corpus), "chunks |", len(queries), "fixed queries |", len(qa), "QA questions")
from collections import Counter
print("by source:", Counter(c["source"] for c in corpus))
print("by lang:  ", Counter(c["lang"] for c in corpus))

OPENAI_API_KEY set: True
HF_TOKEN set:       True
74 chunks | 8 fixed queries | 6 QA questions
by source: Counter({'wikipedia': 51, 'grant_office_handbook': 23})
by lang:   Counter({'en': 71, 'ru': 2, 'kk': 1})


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

> Lowercase, then `re.findall(r"\w+", text)`. In Python 3, `\w` is Unicode-aware, so Kazakh and Russian words stay whole (`өтінішті`, `грант`). Punctuation and hyphens split tokens, so `GR-305` becomes `gr` + `305`. **No stemming** and **no stop-word removal**: words like `what`, `is` and `the` stay in, and BM25's IDF makes them almost worthless anyway. `k1 = 1.5`, `b = 0.75`.

In [3]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in corpus], k1=1.5, b=0.75)

def first_line(text, n=80):
    t = text.split("\n")[0]
    return t if len(t) <= n else t[:n] + "…"

def bm25_search(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    idx = scores.argsort()[::-1][:k]
    return [(corpus[i]["id"], float(scores[i])) for i in idx]

print(tokenize("What is form GR-305 used for?"))
print(bm25_search("What is form GR-305 used for?"))

['what', 'is', 'form', 'gr', '305', 'used', 'for']
[('GO-08', 15.825491699499644), ('GO-12', 8.195256931904641), ('GO-06', 8.07003113015204)]


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Gated model: accept the Gemma licence on its HF page first; HF_TOKEN comes from .env / Colab Secrets.
emb_model = SentenceTransformer("google/embeddinggemma-300m", token=os.environ.get("HF_TOKEN"))  # float32 by default

doc_emb = emb_model.encode_document([c["text"] for c in corpus], convert_to_tensor=False)
doc_emb = np.asarray(doc_emb)
print("document embeddings:", doc_emb.shape, doc_emb.dtype)   # stored as a plain NumPy array in memory

def emb_search(query, k=3):
    q = np.asarray(emb_model.encode_query([query], convert_to_tensor=False))
    sims = np.asarray(emb_model.similarity(q, doc_emb))[0]       # cosine similarity
    idx = sims.argsort()[::-1][:k]
    return [(corpus[i]["id"], float(sims[i])) for i in idx]

print(emb_search("What is form GR-305 used for?"))

c:\Users\shega\css4007-hw3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\shega\css4007-hw3\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shega\.cache\huggingface\hub\models--google--embeddinggemma-300m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate devel

document embeddings: (74, 768) float32
[('GO-08', 0.6955163478851318), ('GO-12', 0.4608723223209381), ('GO-23', 0.4504862129688263)]


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | `GR-210 missing signature` | GO-07 | **BM25.** It is a short keyword query with an exact form code. GO-07 is the only chunk that contains `GR-210`, `missing` and `signature` together. The handbook has three forms with near-identical names (GR-104, GR-210, GR-305), and an embedding model is likely to treat them as "some form code", so it may blur `210` into `104` or `305`. |
| Q-10 | `Что будет с грантом, если я переведусь в другой университет?` | GO-18 | **Embeddings.** The question is in Russian, and the answer (GO-18, "transfers to another university loses it") is only in English. BM25 needs shared tokens and has none: the only Russian chunks it can match are GO-15 and GO-23, and they are about something else. EmbeddingGemma is multilingual, so `переведусь в другой университет` should land near `transfers to another university`. |

In [5]:
my_queries = [
    {"id": "Q-09", "text": "GR-210 missing signature", "relevant": ["GO-07"],
     "expected": "BM25"},
    {"id": "Q-10", "text": "Что будет с грантом, если я переведусь в другой университет?", "relevant": ["GO-18"],
     "expected": "Embeddings"},
]
all_queries = queries + my_queries
for q in all_queries:
    print(q["id"], q["text"], "->", q["relevant"])

Q-01 What is form GR-305 used for? -> ['GO-08', 'GO-12', 'GO-23']
Q-02 My family is poor. Is there financial help for students like me? -> ['GO-01', 'GO-02', 'GO-03']
Q-03 Өтінішті қашанға дейін тапсыру керек? -> ['GO-09', 'GO-14']
Q-04 What is the minimum GPA for the grant? -> ['GO-02']
Q-05 When was Astana renamed Nur-Sultan, and why? -> ['KZ-05']
Q-06 Where did the Soviet Union test its nuclear weapons? -> ['KZ-37', 'KZ-38']
Q-07 Can the grant be sent to my mother's card? -> ['GO-15']
Q-08 If my grades drop in the autumn, do I still get paid in February? -> ['GO-17']
Q-09 GR-210 missing signature -> ['GO-07']
Q-10 Что будет с грантом, если я переведусь в другой университет? -> ['GO-18']


## Results for all ten or more queries: top 3 from each retriever

In [6]:
from IPython.display import Markdown, display

def hit_at_k(results, relevant):
    return any(cid in relevant for cid, _ in results)

def show_top(results):
    for r, (cid, s) in enumerate(results, 1):
        print(f"   {r}. {cid:6s} {s:8.4f}  {first_line(by_id[cid]['text'])}")

results_B = {}
for q in all_queries:
    b = bm25_search(q["text"])
    e = emb_search(q["text"])
    results_B[q["id"]] = {"bm25": b, "emb": e,
                          "bm25_hit": hit_at_k(b, q["relevant"]),
                          "emb_hit": hit_at_k(e, q["relevant"])}
    print("=" * 100)
    print(f"{q['id']}: {q['text']}    relevant={q['relevant']}")
    print(f"  BM25            hit@3 = {results_B[q['id']]['bm25_hit']}")
    show_top(b)
    print(f"  EmbeddingGemma  hit@3 = {results_B[q['id']]['emb_hit']}")
    show_top(e)

Q-01: What is form GR-305 used for?    relevant=['GO-08', 'GO-12', 'GO-23']
  BM25            hit@3 = True
   1. GO-08   15.8255  Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal f…
   2. GO-12    8.1953  A student who disagrees with a decision may appeal within 14 calendar days of th…
   3. GO-06    8.0700  Form GR-104 is the application form for the need-based study grant. It is the on…
  EmbeddingGemma  hit@3 = True
   1. GO-08    0.6955  Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal f…
   2. GO-12    0.4609  A student who disagrees with a decision may appeal within 14 calendar days of th…
   3. GO-23    0.4505  Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в…
Q-02: My family is poor. Is there financial help for students like me?    relevant=['GO-01', 'GO-02', 'GO-03']
  BM25            hit@3 = False
   1. KZ-17    8.7262  The lake currently covers about 16,400 km2 (6,300 sq mi). However,

## B3 — comparison

The table below is printed by the next cell from `results_B`, so it always matches the run above.

In [7]:
def ids_scores(res, digits=2):
    return ", ".join(f"{cid} ({s:.{digits}f})" for cid, s in res)

rows = ["| Query | BM25 top 3 (ids, scores) | BM25 hit@3 | Embedding top 3 (ids, cosine) | Embedding hit@3 |",
        "|---|---|:---:|---|:---:|"]
for q in all_queries:
    r = results_B[q["id"]]
    rows.append(f"| {q['id']} | {ids_scores(r['bm25'], 2)} | {'✓' if r['bm25_hit'] else '✗'} "
                f"| {ids_scores(r['emb'], 3)} | {'✓' if r['emb_hit'] else '✗'} |")
n = len(all_queries)
bm_total = sum(r["bm25_hit"] for r in results_B.values())
em_total = sum(r["emb_hit"] for r in results_B.values())
rows.append(f"| **hit@3 total** | | **{bm_total}/{n}** | | **{em_total}/{n}** |")
display(Markdown("\n".join(rows)))

| Query | BM25 top 3 (ids, scores) | BM25 hit@3 | Embedding top 3 (ids, cosine) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08 (15.83), GO-12 (8.20), GO-06 (8.07) | ✓ | GO-08 (0.696), GO-12 (0.461), GO-23 (0.450) | ✓ |
| Q-02 | KZ-17 (8.73), GO-20 (8.31), KZ-25 (7.51) | ✗ | GO-01 (0.445), GO-10 (0.412), GO-20 (0.409) | ✓ |
| Q-03 | GO-14 (10.70), GO-23 (0.00), GO-22 (0.00) | ✓ | GO-14 (0.399), GO-09 (0.350), GO-11 (0.301) | ✓ |
| Q-04 | GO-05 (12.78), GO-17 (10.52), GO-19 (10.14) | ✗ | GO-02 (0.665), GO-05 (0.621), GO-17 (0.613) | ✓ |
| Q-05 | KZ-05 (15.73), KZ-06 (6.44), KZ-11 (5.87) | ✓ | KZ-05 (0.651), KZ-06 (0.507), KZ-03 (0.419) | ✓ |
| Q-06 | KZ-37 (19.09), KZ-38 (17.17), KZ-03 (9.35) | ✓ | KZ-37 (0.596), KZ-38 (0.510), KZ-10 (0.331) | ✓ |
| Q-07 | GO-21 (14.65), GO-16 (9.16), GO-01 (9.05) | ✗ | GO-15 (0.422), GO-18 (0.417), GO-01 (0.411) | ✓ |
| Q-08 | GO-17 (17.17), GO-05 (12.77), GO-04 (11.93) | ✓ | GO-17 (0.559), GO-18 (0.459), GO-15 (0.453) | ✓ |
| Q-09 | GO-07 (15.19), GO-11 (5.28), GO-13 (5.03) | ✓ | GO-07 (0.632), GO-10 (0.428), GO-13 (0.380) | ✓ |
| Q-10 | GO-23 (15.09), GO-15 (10.95), GO-22 (0.00) | ✗ | GO-18 (0.690), GO-15 (0.544), GO-17 (0.531) | ✓ |
| **hit@3 total** | | **6/10** | | **10/10** |

**Both retrievers agree:** which query, and why was it easy?

> **Q-06** ("Where did the Soviet Union test its nuclear weapons?"). Both put the two Semipalatinsk chunks first and second: BM25 KZ-37 (19.09) and KZ-38 (17.17), EmbeddingGemma KZ-37 (0.596) and KZ-38 (0.510). It was easy because the query and the chunks share rare, specific words (`Soviet`, `nuclear`, `test`, `weapons`), which get a high IDF in BM25. The meaning is also unambiguous, so the query vector lands right next to those chunks. Q-01 (`GR-305` → GO-08 first for both, 15.83 and 0.696) and Q-05 (`Nur-Sultan` → KZ-05 first for both, 15.73 and 0.651) are easy for the same reason.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> In my run, **BM25 never beat EmbeddingGemma on hit@3**. Embeddings got 10/10, BM25 6/10. Where BM25 was better is in *how sharply* it separated the right chunk from the rest when the query had an exact rare string:
> - **Q-09** ("GR-210 missing signature"): BM25 GO-07 **15.19**, then GO-11 5.28. The right chunk scores 2.9× the runner-up, because only GO-07 contains `210` *and* `signature`. EmbeddingGemma also ranked GO-07 first, but at 0.632 against GO-10 at 0.428, a ratio of only 1.5. The exact code `210` and the word `signature` are blurred into "something about the income form".
> - **Q-03** (Kazakh): BM25 found GO-14 at **10.70**, and every other chunk scored **0.00**, because only GO-14 shares the exact Kazakh tokens. EmbeddingGemma found GO-14 (0.399) and GO-09 (0.350), but put GO-11 (*When a document is missing*, 0.301) third, which has nothing to do with deadlines.
>
> So my prediction for Q-09 was half right: BM25 matched the exact string best, but the embedding model was good enough to still rank it first.

**Embeddings did better:** which query, and what could BM25 not match?

> Four queries: Q-02, Q-04, Q-07 and Q-10. EmbeddingGemma hit all four and BM25 missed all four.
> - **Q-02** ("My family is poor. Is there financial help…"): a **synonym/paraphrase** case. The handbook never says `poor` or `financial help`; it says `low-income household` and `need-based grant`. BM25 ranked **KZ-17 Lake Balkhash** first (8.73) on function words alone: `like` 2.95, `there` 2.22, `is` ×2 ≈ 3.4. That is the cost of my choice to keep stop words. EmbeddingGemma ranked **GO-01** first (0.445).
> - **Q-07** ("Can the grant be sent to my mother's card?"): **another language**. The answer is only in Russian in GO-15 (`перевод на счёт родителей … невозможен`). BM25 ranked GO-21 first (14.65), because GO-21 says *"Questions **can** also **be sent** through…"*: `sent` 4.39 and `can` 3.94. EmbeddingGemma ranked **GO-15 first** (0.422), matching `mother's card` to `счёт родителей` across languages.
> - **Q-10** (my embedding query, Russian question with the answer in English GO-18): BM25 returned only the two Russian chunks, GO-23 (15.09) and GO-15 (10.95), matched on Russian function words `в` (5.73), `с` (5.02) and `если` (4.34). EmbeddingGemma put **GO-18 first with 0.690**, its highest score on any handbook query. My prediction was right.
> - **Q-04** ("What is the minimum GPA for the grant?"): BM25 ranked **GO-05** first (12.78). Its title is *"Archived: the 2025 scheme (no longer in force)"*. The deciding word is `minimum` (4.57 of the score), which GO-05 contains (*"the minimum GPA for the grant was 3.0"*) and the current rule GO-02 does not. GO-02 is not in BM25's top 3 at all. EmbeddingGemma ranked **GO-02 first (0.665)**, but only just: the archived GO-05 is second at 0.621. Neither retriever knows that "archived" means "do not use". Embeddings won here on meaning (`minimum GPA` ≈ `GPA is at least`), not because they understood the archive.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [8]:
from openai import OpenAI
import tiktoken

client = OpenAI()          # reads OPENAI_API_KEY from the environment; the key is never printed
MODEL = "gpt-5.6-luna"
enc = tiktoken.get_encoding("o200k_base")

SYSTEM = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)
SYSTEM_LLM_ONLY = "Answer the question."   # baseline: no context, no grounding instruction

def ntok(s):
    return len(enc.encode(s))

def build_context(chunk_ids):
    blocks = []
    for i, cid in enumerate(chunk_ids, 1):
        c = by_id[cid]
        blocks.append(f"[{i}] ({cid}) {c['title']}\n{c['text']}")
    return "\n\n".join(blocks)

def build_messages(question, chunk_ids=None):
    if chunk_ids is None:
        return [{"role": "system", "content": SYSTEM_LLM_ONLY},
                {"role": "user", "content": question}]
    user = f"Context:\n\n{build_context(chunk_ids)}\n\nQuestion: {question}"
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]

def ask(messages):
    resp = client.chat.completions.create(model=MODEL, messages=messages)
    return resp.choices[0].message.content, resp.usage

def retrieve(question, method, k=3):
    return (bm25_search if method == "bm25" else emb_search)(question, k)

## C1 — the complete prompt, printed before it is sent

In [9]:
c1 = next(q for q in qa if q["id"] == "C-01")
c1_ret = emb_search(c1["question"], k=3)
c1_ids = [cid for cid, _ in c1_ret]
c1_msgs = build_messages(c1["question"], c1_ids)

print("Retrieved (EmbeddingGemma):", [(cid, round(s, 4)) for cid, s in c1_ret])
print()
for m in c1_msgs:
    print(f"======== {m['role'].upper()} ========")
    print(m["content"])
    print()

Retrieved (EmbeddingGemma): [('GO-04', 0.5845), ('GO-03', 0.4842), ('GO-15', 0.4506)]

======== SYSTEM ========
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

======== USER ========
Context:

[1] (GO-04) How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] (GO-03) Income bands
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,00

In [10]:
# Token accounting with tiktoken o200k_base, then the real call.
context_str = build_context(c1_ids)
user_str = c1_msgs[1]["content"]
t_system = ntok(SYSTEM)
t_question = ntok(c1["question"])
t_context = ntok(context_str)
t_wrapper = ntok(user_str) - t_question - t_context   # "Context:", "Question:", blank lines
t_total = t_system + ntok(user_str)

c1_answer, c1_usage = ask(c1_msgs)

per_chunk = {cid: ntok(by_id[cid]["text"]) for cid in c1_ids}
display(Markdown(f'''
| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | {t_system} |
| Question | {t_question} |
| Retrieved context (3 numbered blocks incl. ids and titles) | {t_context} |
| Template words ("Context:", "Question:", separators) | {t_wrapper} |
| **Total input** (sum of message contents) | **{t_total}** |
| **`usage.prompt_tokens` from the response** | **{c1_usage.prompt_tokens}** |
| Difference | {c1_usage.prompt_tokens - t_total} |
'''))
print("Number of retrieved chunks:", len(c1_ids))
print("Tokens per chunk (text only):", per_chunk)
print(f"Total context size: {t_context} tokens / {len(context_str)} characters")
print()
print("ANSWER:\n", c1_answer)


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context (3 numbered blocks incl. ids and titles) | 317 |
| Template words ("Context:", "Question:", separators) | 4 |
| **Total input** (sum of message contents) | **383** |
| **`usage.prompt_tokens` from the response** | **393** |
| Difference | 10 |


Number of retrieved chunks: 3
Tokens per chunk (text only): {'GO-04': 90, 'GO-03': 104, 'GO-15': 85}
Total context size: 317 tokens / 1221 characters

ANSWER:
 A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in February**. [1]


- Number of retrieved chunks: **3** (GO-04 0.5845, GO-03 0.4842, GO-15 0.4506)
- Tokens per chunk (text only): **GO-04 90, GO-03 104, GO-15 85**
- Total context size (tokens / characters): **317 tokens / 1221 characters**

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> My tiktoken sum of the message texts is **383**, and the API reported **`usage.prompt_tokens` = 393**, so **10 tokens more**. tiktoken counts only the text inside my two messages. The API does not send bare text: it wraps each message in the chat format, with special tokens marking the start of a message, its role (`system`, `user`) and its end, plus a few tokens that open the assistant's reply. Those tokens are billed but are not in my strings. With 2 messages, about 3–4 tokens each plus about 3 for the reply primer gives roughly 10. This is the same gap as in HW1: the API's count is always slightly above the count of the visible text.

**What exactly does retrieval add to the model's input?**

> Literally: the three chunk texts EmbeddingGemma ranked highest (GO-04, GO-03, GO-15), pasted as plain strings into the user message, each one numbered `[1]`–`[3]` with its id and title, after the word `Context:`. That is **317 of the 383 tokens (83%)**. The rest is the 44-token system instruction, the 18-token question and 4 tokens of template. No vectors, scores, URLs or files reach the model, and it does not open the corpus. Note that one of the three blocks (GO-15) is in Russian. The model got it as-is, and it still answered in English from block [1]. The answer even contains something that is in no block: *"75,000 tenge in October and 75,000 tenge in February"*. The model computed this from *"two equal instalments"*. It is correct, but it came from generation, not from the text.

## C2 — three systems, six questions

In [11]:
SYSTEMS = ["llm", "bm25", "emb"]
results_C = {}
for q in qa:
    results_C[q["id"]] = {}
    for sysname in SYSTEMS:
        if sysname == "llm":
            ids, ret = None, None
        else:
            ret = retrieve(q["question"], sysname, k=3)
            ids = [cid for cid, _ in ret]
        msgs = build_messages(q["question"], ids)
        ans, usage = ask(msgs)
        results_C[q["id"]][sysname] = {"retrieved": ret, "messages": msgs,
                                       "answer": ans, "prompt_tokens": usage.prompt_tokens}
    print("=" * 100)
    print(f"{q['id']}: {q['question']}")
    print(f"REFERENCE: {q['reference']}   (relevant: {q['relevant']})")
    for sysname in SYSTEMS:
        r = results_C[q["id"]][sysname]
        got = "" if r["retrieved"] is None else "  retrieved=" + ", ".join(f"{c} ({s:.3f})" for c, s in r["retrieved"])
        print(f"\n--- {sysname.upper()}{got}  [prompt_tokens={r['prompt_tokens']}]")
        print(r["answer"])

C-01: How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.   (relevant: ['GO-04', 'GO-15'])

--- LLM  [prompt_tokens=32]
Which student funding scheme and academic year do you mean? “Income band 2” varies by country and program, so I need that context to provide the amount and payment date.

--- BM25  retrieved=GO-04 (20.219), GO-02 (18.742), GO-01 (17.243)  [prompt_tokens=401]
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]

--- EMB  retrieved=GO-04 (0.585), GO-03 (0.484), GO-15 (0.451)  [prompt_tokens=393]
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: the first in **October** and the second in **February**. [1]
C-02: My grant application was refused. How long do I have 

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

**How I marked.** I read every answer above. The `MARKS` dictionary in the next cell is filled in by hand, and the cell prints the table from it. A correct answer in different words is ✓. The right number with an invented form code is ✗. For C-06, the only ✓ is "I don't know / the handbook does not say / master's students have a separate scheme". Any number is ✗.

In [17]:
# ✓ / ✗ — marked by hand after reading the answers above (agrees with `reference`).
MARKS = {
    # LLM only: asked which scheme | BM25: 150,000, Oct + Feb | Emb: same, plus 75,000 per instalment (correct arithmetic)
    "C-01": {"llm": "✗", "bm25": "✓", "emb": "✓"},
    # LLM only: "often 14, 21 or 28 days", no form | both RAG: 14 calendar days, GR-305
    "C-02": {"llm": "✗", "bm25": "✓", "emb": "✓"},
    # LLM only: asked which grant | both RAG: 2.67 on 4.0 (Emb ignored archived 3.0 in its block [3])
    "C-03": {"llm": "✗", "bm25": "✓", "emb": "✓"},
    # all three: Oct 2017 and 2031 (none mentions that the 2031 date was set in Jan 2021)
    "C-04": {"llm": "✓", "bm25": "✓", "emb": "✓"},
    # all three: 1963, Almaty (Alma-Ata) Institute of National Economy
    "C-05": {"llm": "✓", "bm25": "✓", "emb": "✓"},
    # LLM only: US Pell-grant advice, never says the handbook has no answer | both RAG: not eligible, separate scheme, ask department
    "C-06": {"llm": "✗", "bm25": "✓", "emb": "✓"},
}

def short(ans, n=90):
    a = " ".join(ans.split()).replace("|", "/")
    return a if len(a) <= n else a[:n] + "…"

rows = ["| Question | LLM only | BM25 RAG | Embedding RAG |", "|---|---|---|---|"]
for q in qa:
    cells_ = [f"{MARKS[q['id']][s]} {short(results_C[q['id']][s]['answer'])}" for s in SYSTEMS]
    rows.append(f"| {q['id']} | " + " | ".join(cells_) + " |")
totals = {s: sum(MARKS[q["id"]][s] == "✓" for q in qa) for s in SYSTEMS}
rows.append(f"| **agrees with `reference`** | **{totals['llm']}/6** | **{totals['bm25']}/6** | **{totals['emb']}/6** |")
display(Markdown("\n".join(rows)))

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ Which student funding scheme and academic year do you mean? “Income band 2” varies by coun… | ✓ A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two… | ✓ A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two… |
| C-02 | ✗ The deadline and appeal form depend on the **grant scheme and country/agency**. Check your… | ✓ You have **14 calendar days from the date on the decision letter** to appeal. Use **Form G… | ✓ You have **14 calendar days from the date on the decision letter** to appeal. Use **form G… |
| C-03 | ✗ Which need-based grant or institution are you referring to? The minimum GPA varies by prog… | ✓ The minimum GPA is **2.67 on a 4.0 scale**. [2] | ✓ The minimum GPA needed to qualify for the 2026 need-based grant is **2.67 on a 4.0 scale**… |
| C-04 | ✓ Kazakhstan decided to switch from the Cyrillic alphabet to a Latin-based Kazakh alphabet i… | ✓ Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The deadline was … | ✓ Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition is… |
| C-05 | ✓ Narxoz University was founded in **1963** as the **Alma-Ata Institute of National Economy*… | ✓ Narxoz University was founded in **1963** as the **Almaty Institute of National Economy**.… | ✓ Narxoz University was founded in **1963** as the **Almaty Institute of National Economy**.… |
| C-06 | ✗ There isn’t a universal amount—it depends on your country, university, program, financial … | ✓ You cannot apply for this need-based grant because it is only for full-time undergraduate … | ✓ You cannot apply for this need-based grant because it is only for full-time undergraduate … |
| **agrees with `reference`** | **2/6** | **6/6** | **6/6** |

## C3 — question 1 of 3: C-03

The code cell below prints everything for this question from the C2 run.

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (15.7985, *Combining the grant with other support*), **GO-02 (14.0587, *Who qualifies*)**, GO-06 (13.8516, *Form GR-104*). The archived GO-05 is just outside, at rank 4 with 13.83.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> **GO-02 (0.6286)**, GO-20 (0.5691, *Who cannot apply*), **GO-05 (0.5451, *Archived: the 2025 scheme*)**.

**3. What was sent to the model** (the context blocks):

> - **BM25 RAG:** [1] GO-16, [2] GO-02, [3] GO-06. Only [2] has a GPA: *"their cumulative GPA is at least 2.67 on the 4.0 scale"*.
> - **Embedding RAG:** [1] GO-02 (*"at least 2.67"*), [2] GO-20, [3] **GO-05**: *"ARCHIVED … Under the 2025 scheme the minimum GPA for the grant was 3.0 … do not apply to any application made for the 2026 academic year"*. So two contradicting numbers were in the same prompt.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> LLM only did not guess. It asked back: *"Which need-based grant or institution are you referring to? The minimum GPA varies by program."* (✗, no answer). Both RAG systems became **specific and grounded**. BM25 RAG answered *"The minimum GPA is 2.67 on a 4.0 scale. [2]"*. Embedding RAG answered *"…for the 2026 need-based grant is 2.67 on a 4.0 scale. [1]"*: it even added "2026", which shows it used the year to pick GO-02 over GO-05.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed at K = 3, and this went better than I expected. I expected the trap to work: the Embedding RAG prompt contained the archived *"minimum GPA for the grant was 3.0"* in block [3], worded almost exactly like the question. Retrieval did let the trap in, which is a retrieval weakness, since GO-05 should never be sent. But **generation handled it**: the model cited only [1] and ignored the 3.0. The retrieval failure appears when K = 1 (final question 8): BM25 sends only GO-16, and the model answers *"I do not know. The provided context does not specify a minimum GPA requirement."* The 2.67 fact was not in the context, so that is a **retrieval failure**, and the model correctly refused instead of inventing a number.

In [13]:
qid = "C-03"
q = next(x for x in qa if x["id"] == qid)
print(q["question"]); print("REFERENCE:", q["reference"], "| relevant:", q["relevant"]); print()
for sysname in ["bm25", "emb"]:
    r = results_C[qid][sysname]
    print(f"{sysname.upper()} top 3:", ", ".join(f"{c} ({s:.4f})" for c, s in r["retrieved"]))
print()
for sysname in SYSTEMS:
    r = results_C[qid][sysname]
    print("=" * 30, sysname.upper(), "— what was sent", "=" * 30)
    for m in r["messages"]:
        print(f"[{m['role']}]"); print(m["content"]); print()
    print("ANSWER:", r["answer"]); print()

What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.) | relevant: ['GO-02']

BM25 top 3: GO-16 (15.7985), GO-02 (14.0587), GO-06 (13.8516)
EMB top 3: GO-02 (0.6286), GO-20 (0.5691), GO-05 (0.5451)

============================== LLM — what was sent ==============================
[system]
Answer the question.

[user]
What is the minimum GPA needed to qualify for the need-based grant?

ANSWER: Which need-based grant or institution are you referring to? The minimum GPA varies by program.

============================== BM25 — what was sent ==============================
[system]
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

[user]
Context:

[1] (GO-16) Combining the grant with other support
The need-based grant can be held together with

## C3 — question 2 of 3: C-06

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (16.9958), GO-01 (14.1725), **GO-20 (13.7202, *Who cannot apply*)**.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> **GO-20 (0.6664)**, GO-16 (0.6161), GO-01 (0.5539). These are the same three chunks as BM25, but with GO-20 first instead of third.

**3. What was sent to the model** (the context blocks):

> Both RAG prompts contained GO-20 (*"Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules"*), GO-16 (*"capped at 400,000 tenge"*) and GO-01 (*"for full-time undergraduate students"*). The only difference is the order: GO-20 is block [3] for BM25 and block [1] for EmbeddingGemma.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> This is the one question with no answer in the corpus. LLM only answered a different question: it gave **US** advice (*"Federal Pell Grants generally are not available for regular master's programs … $0 to several thousand dollars per year"*). It never says the Grant Office handbook has no rule for master's students (✗). Both RAG systems gave the one correct response, almost word for word: *"You cannot apply for this need-based grant because it is only for full-time undergraduate students. Master's students are covered by a separate scheme and should ask their department…"*, citing GO-20 ([3] for BM25, [1] for embeddings). Retrieval made the answer **grounded**, and it turned "here are some numbers" into a correct "not covered here".

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed. Retrieval succeeded for both, because GO-20 *is* the evidence that there is no answer. Generation also succeeded: the context contained a tempting number (*"capped at 400,000 tenge"* in GO-16), and neither model attached it to a master's student. BM25 had GO-20 only in third place behind two less useful chunks. With K = 2 it would have dropped out, and the model would have had only GO-16 and GO-01. Those two say the grant is for undergraduates, but they say nothing about master's students' own scheme.

In [14]:
qid = "C-06"
q = next(x for x in qa if x["id"] == qid)
print(q["question"]); print("REFERENCE:", q["reference"], "| relevant:", q["relevant"]); print()
for sysname in ["bm25", "emb"]:
    r = results_C[qid][sysname]
    print(f"{sysname.upper()} top 3:", ", ".join(f"{c} ({s:.4f})" for c, s in r["retrieved"]))
print()
for sysname in SYSTEMS:
    r = results_C[qid][sysname]
    print("=" * 30, sysname.upper(), "— what was sent", "=" * 30)
    for m in r["messages"]:
        print(f"[{m['role']}]"); print(m["content"]); print()
    print("ANSWER:", r["answer"]); print()

I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe. | relevant: ['GO-20']

BM25 top 3: GO-16 (16.9958), GO-01 (14.1725), GO-20 (13.7202)
EMB top 3: GO-20 (0.6664), GO-16 (0.6161), GO-01 (0.5539)

============================== LLM — what was sent ==============================
[system]
Answer the question.

[user]
I am a master's student. How much need-based grant can I get?

ANSWER: There isn’t a universal amount—it depends on your country, university, program, financial need, and enrollment status.

**If you mean the United States:**  
- **Federal Pell Grants** generally are **not available for regular master’s programs**. A limited exception may apply to certain post-baccalaureate teacher-certification programs.  
- Graduate students may instead qualify for **university/state grants, scholarships, fellowships, assistantships, or Federal Work-Study**.

## C3 — question 3 of 3: C-02

**1. What BM25 retrieved** (top 3, ids and scores):

> **GO-05 (17.2414, *Archived: the 2025 scheme*)**, GO-08 (14.5287, *Form GR-305: appeals*), GO-12 (13.5735, *How to appeal a decision*).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-08 (0.6029), GO-12 (0.5591), **GO-11 (0.5343, *When a document is missing*)**.

**3. What was sent to the model** (the context blocks):

> - **BM25 RAG:** [1] **GO-05**, the archived 2025 rules, with nothing about appeals. It ranked first because of `do` (4.57, from *"do not apply"*) and `application` (2.98). [2] GO-08 (*"Form GR-305 is used to appeal a decision"*). [3] GO-12 (*"may appeal within 14 calendar days of the date on the decision letter, using form GR-305"*).
> - **Embedding RAG:** [1] GO-08, [2] GO-12, [3] **GO-11**, which is about missing documents and contains a distracting number: *"The student then has **10 working days** to upload it"*.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> LLM only could not know the handbook. It said the deadline is *"often 14, 21, or 28 days"* and that the form depends on the agency (✗: no form code, no single deadline). Both RAG systems became **specific and grounded**: *"14 calendar days from the date on the decision letter … Form GR-305. [2][3]"* (BM25) and *"…form GR-305, the Grant Office appeal form. [2]"* (embeddings). The form code and "calendar" (not "working") days exist only in the chunks, so this is the clearest example of what retrieval adds.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing in the reference was missed. Both retrievers wasted one of three slots on an off-topic chunk: archived GO-05 for BM25 and GO-11 for embeddings. That is a partial retrieval miss. **Generation handled it**: neither model used GO-05's 2025 rules or GO-11's *"10 working days"*, and both cited only the appeal blocks.

**A RAG answer that was worse than I expected: C-04 (from the C2 output).** No RAG answer was marked ✗ in my run, but on C-04 both RAG answers were *less complete than LLM only*. LLM only said *"October 26, 2017 … The original target was 2025, but the timetable was later extended … 2031"*. Both RAG prompts contained KZ-43 (BM25 rank 1, 25.966; EmbeddingGemma rank 2, 0.693), which says *"…to be implemented by 2025. In January 2021, the target year for finishing the transition was pushed back to 2031."* Yet Embedding RAG answered only *"October 2017 … by 2031"*. BM25 RAG answered *"October 2017. The deadline was later postponed … 2031"*, without the original 2025 target or the January 2021 date. The fact was **in the context**, so this is a **generation** shortfall, not a retrieval one. The model answered the minimum the question literally asked and dropped details it had been given.

In [15]:
qid = "C-02"
q = next(x for x in qa if x["id"] == qid)
print(q["question"]); print("REFERENCE:", q["reference"], "| relevant:", q["relevant"]); print()
for sysname in ["bm25", "emb"]:
    r = results_C[qid][sysname]
    print(f"{sysname.upper()} top 3:", ", ".join(f"{c} ({s:.4f})" for c, s in r["retrieved"]))
print()
for sysname in SYSTEMS:
    r = results_C[qid][sysname]
    print("=" * 30, sysname.upper(), "— what was sent", "=" * 30)
    for m in r["messages"]:
        print(f"[{m['role']}]"); print(m["content"]); print()
    print("ANSWER:", r["answer"]); print()

My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305. | relevant: ['GO-08', 'GO-12', 'GO-23']

BM25 top 3: GO-05 (17.2414), GO-08 (14.5287), GO-12 (13.5735)
EMB top 3: GO-08 (0.6029), GO-12 (0.5591), GO-11 (0.5343)

============================== LLM — what was sent ==============================
[system]
Answer the question.

[user]
My grant application was refused. How long do I have to appeal, and which form do I use?

ANSWER: The deadline and appeal form depend on the **grant scheme and country/agency**. Check your refusal letter or decision email first—it should state:

- the appeal/review deadline, often **14, 21, or 28 days** from the decision date; and  
- the required form or online appeal portal.

If the letter does not say, contact the grant administrator immediately and ask for the **appeal/reconsideration procedure**. Tell me the **grant name, issuing bod

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 scores a chunk by counting exact token overlaps with the query, weighted by how rare each token is (IDF) and normalised by chunk length. Embedding retrieval turns the query and every chunk into vectors and ranks by the angle between them, so it compares meaning rather than spelling. Part A showed it directly: BM25 gave D2 a score of 0 for "RAG retrieve" against "Retrieval-Augmented … retrieves", while the vectors still put D2 first.

**2. When can BM25 perform better than embeddings?**

> When the query contains a rare exact string, such as a form code, an id, a name or a number, and when the result must be explainable token by token. In my run BM25 never won on hit@3, but it separated exact matches much more sharply. Q-09 `GR-210`: GO-07 at 15.19 against a runner-up of 5.28, while embeddings gave 0.632 against 0.428. Q-03 (Kazakh): GO-14 at 10.70, everything else 0.00. BM25 also needs no model download, no GPU and no gated licence.

**3. When can embeddings perform better than BM25?**

> When the query says the same thing in other words or in another language. In my run embeddings won all four queries that BM25 missed. Q-02 is a paraphrase ("poor" ≈ "low-income household": GO-01 at 0.445, while BM25 picked Lake Balkhash). Q-07 and Q-10 are cross-language (English → Russian GO-15 at 0.422, Russian → English GO-18 at 0.690). On Q-04, `minimum GPA` ≈ `GPA is at least` gave GO-02 at 0.665, while BM25 picked the archived GO-05.

**4. Why is cosine similarity used with embeddings?**

> Cosine measures the angle between two vectors and ignores their length. In an embedding, the direction carries the meaning, and the length mostly reflects things like text length. So two texts about the same topic score high even if one is much longer. It is also cheap (a dot product) and bounded in [−1, 1]. My scores ranged from 0.27 (unrelated) to 0.72 (KZ-14 for C-04), which makes them easy to compare.

**5. What information is actually sent to an LLM in a RAG system?**

> Only text: the system instruction (44 tokens), the top-K chunk texts with `[n]`, id and title (317 tokens for C-01), and the question (18 tokens). Then the API adds about 10 formatting tokens (383 → 393 `prompt_tokens`). Vectors, scores and the other 71 chunks are never sent.

**6. Does the LLM search the original documents itself?**

> No. My code ranks the chunks and pastes three of them in. The model only continues the prompt. With K = 1 on C-03, it was given only GO-16 and said *"I do not know"*, even though the answer was sitting in GO-02 in the same corpus. It could not go and look.

**7. What happens if retrieval returns an irrelevant chunk?**

> It costs a slot and tokens, and it can mislead. In my run the model usually ignored it: archived GO-05 was BM25's block [1] for C-02, GO-11 (*"10 working days"*) was embeddings' block [3] for C-02, and GO-05's *"3.0"* was embeddings' block [3] for C-03. None of them leaked into an answer. The real damage comes when an irrelevant chunk *pushes out* the right one, as at K = 1 for C-03.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> C-03 with BM25 (output in the next cell):
> - **K = 1:** only GO-16 (15.8), 158 prompt tokens. Answer: *"I do not know. The provided context does not specify a minimum GPA requirement."* Retrieval failure.
> - **K = 3:** GO-02 enters at rank 2 (14.06), 371 tokens. Answer: *"2.67 on a 4.0 scale [2]"*. Correct.
> - **K = 8:** 900 tokens (2.4× K = 3). Now the archived GO-05 (rank 4, 13.83, *"minimum GPA … was 3.0"*) and five more chunks are included. The answer stayed *"2.67 [2]"*, so the model resolved the contradiction, but I paid 2.4× the input for no gain.
>
> A larger K raises recall (K = 1 → 3 fixed the answer). It also adds cost, noise and contradicting chunks that the model has to sort out (K = 8). Here it survived, but a weaker model or a less clearly labelled archive chunk might not.

In [16]:
qid = "C-03"
q = next(x for x in qa if x["id"] == qid)
for k in (1, 3, 8):
    ret = bm25_search(q["question"], k=k)
    ids = [cid for cid, _ in ret]
    ans, usage = ask(build_messages(q["question"], ids))
    print(f"===== K={k}  retrieved={[(c, round(s, 2)) for c, s in ret]}  prompt_tokens={usage.prompt_tokens}")
    print(ans); print()

===== K=1  retrieved=[('GO-16', 15.8)]  prompt_tokens=158
I do not know. The provided context does not specify a minimum GPA requirement.

===== K=3  retrieved=[('GO-16', 15.8), ('GO-02', 14.06), ('GO-06', 13.85)]  prompt_tokens=371
The minimum GPA is **2.67 on a 4.0 scale**. [2]

===== K=8  retrieved=[('GO-16', 15.8), ('GO-02', 14.06), ('GO-06', 13.85), ('GO-05', 13.83), ('GO-01', 13.42), ('GO-11', 12.44), ('GO-03', 11.71), ('GO-17', 11.51)]  prompt_tokens=900
The minimum GPA needed is **2.67 on a 4.0 scale**. [2]



**9. Is a vector database required to build a RAG system?**

> No. My notebook keeps the 74 EmbeddingGemma vectors in a plain NumPy array (`doc_emb`, shape (74, 768), float32) in memory. "Search" is one `emb_model.similarity` call plus `argsort`. BM25 needs no vectors at all. A vector database (FAISS, pgvector, Qdrant…) only becomes useful at millions of chunks, or when you need persistence, filters and updates.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> **EmbeddingGemma.** hit@3 over my 10 queries: **embeddings 10/10, BM25 6/10**. BM25 missed Q-02 (paraphrase), Q-04 (picked archived GO-05), Q-07 and Q-10 (cross-language). C2 agreement with `reference`: **BM25 RAG 6/6, Embedding RAG 6/6, LLM only 2/6**. RAG made the difference, not the choice of retriever: the six C2 questions all contain words that also appear in their answer chunk, so both retrievers got the right chunk into the top 3 every time. Embeddings still ranked it higher: rank 1 vs rank 2 for C-02 and C-03, and rank 1 vs rank 3 for C-06. That leaves more margin if K were smaller. On a corpus with Kazakh and Russian chunks and students who type in their own words, that is the evidence that counts.

---
# Conclusion

> Next I would build **hybrid retrieval with a metadata filter**. First, run BM25 and EmbeddingGemma together and merge their rankings (for example, reciprocal-rank fusion). Embeddings carry the paraphrase and cross-language cases (Q-02, Q-07, Q-10), and BM25 adds sharp exact-code matches (Q-09: 15.19 against 5.28). Second, give every chunk a `status` field and drop archived chunks like GO-05 *before* ranking. It came back in BM25's #1 for Q-04 and C-02 and in embeddings' top 3 for C-03, Q-04 and K = 8. This time the model ignored it, but nothing guarantees that. Finally, I would keep the "say you do not know" instruction and add more C-06-style questions to the test set: the LLM-only answer to C-06 showed how confidently a model fills a gap with someone else's rules.